In [ ]:
import json, os
from google.colab import files

if 'spark_jsl.json' not in os.listdir():
  license_keys = files.upload()
  os.rename(list(license_keys.keys())[0], 'spark_jsl.json')

with open('spark_jsl.json') as f:
    license_keys = json.load(f)

locals().update(license_keys)
os.environ.update(license_keys)

In [ ]:
! pip install --upgrade -q pyspark==3.5.1 spark-nlp==$PUBLIC_VERSION

! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

! pip install -q spark-nlp-display

In [ ]:
%pip install -U typesafe-sdk

In [ ]:
from google.colab import userdata

# The name must exactly match your Colab Secret name.
typesafe_api_key = userdata.get("TYPESAFE_API_KEY")

if not typesafe_api_key:
    raise ValueError(
        "TYPESAFE_API_KEY was not loaded. Check the Secret name and "
        "enable Notebook access for that secret."
    )

typesafe_api_key = typesafe_api_key.strip()
print("TypeSafe key loaded:", bool(typesafe_api_key))

TypeSafe key loaded: True


In [ ]:
import json
import os
import re

import sparknlp
import sparknlp_jsl

from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp_jsl.pipeline_tracer import PipelineTracer
from sparknlp_jsl.pipeline_output_parser import PipelineOutputParser

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline,PipelineModel


import pandas as pd
pd.set_option('display.max_colwidth', 200)

import warnings
warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"25G",
          "spark.kryoserializer.buffer.max":"2000M",
          "spark.driver.maxResultSize":"2000M"}

spark = sparknlp_jsl.start(license_keys['SECRET'],params=params)

print("Spark NLP Version :", sparknlp.version())
print("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark

Spark NLP Version : 6.4.0
Spark NLP_JSL Version : 6.4.0


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! unzip '/content/drive/MyDrive/JSL-Colabs/Cancer/models/ner_cancer_registry_gp_aug_september_21_all.zip' -d '/content/model/ner_cancer_registry_gp_september_21_all'

In [36]:
# -------------------------------------------------------------------
# Spark NLP candidate extraction: NER + split-aligned deterministic rules
# -------------------------------------------------------------------
documentAssembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

document_splitter = InternalDocumentSplitter()\
    .setInputCols("document")\
    .setOutputCol("splits")\
    .setSplitMode("regex")\
    .setSplitPatterns([r"\n\s*\n"])\
    .setPatternsAreRegex(True)\
    .setExplodeSplits(True)\
    .setTrimWhitespace(True)

sentenceDetector = SentenceDetectorDLModel.pretrained(
    "sentence_detector_dl_healthcare", "en", "clinical/models"
).setInputCols(["splits"]).setOutputCol("sentence")

tokenizer = Tokenizer()\
    .setInputCols(["sentence"])\
    .setOutputCol("token")

word_embeddings = WordEmbeddingsModel.pretrained(
    "embeddings_clinical", "en", "clinical/models"
).setInputCols(["sentence", "token"]).setOutputCol("embeddings")

ner = MedicalNerModel.load(
    "/content/model/ner_cancer_registry_gp_september_21_all"
).setInputCols(["sentence", "token", "embeddings"]).setOutputCol("ner")

ner_converter_all = NerConverterInternal()\
    .setInputCols(["sentence", "token", "ner"])\
    .setOutputCol("ner_chunk")

grade_matcher = (
    RegexMatcherInternal()
    .setInputCols(["splits"])
    .setOutputCol("grade_rule_chunk")
    .setRules([
        r"(?i)\bgrade\s+[1-5]\b~Grade",
        r"(?i)\b(?:low|intermediate|high)[-\s]grade\b~Grade",
    ])
    .setDelimiter("~")
    .setStrategy("MATCH_ALL")
)

# Pathology/Test policy for this registry: CBC and CMP remain
# Pathology_Test, even though a later ontology may split generic lab tests.
# Keep generic "biopsy" after longer phrases. ChunkMergeApproach resolves
# nested overlaps when all CHUNK columns are merged below.
pathology_patterns_path = "/content/pathology_patterns.json"
pathology_patterns = [
    {
        "id": "pathology_tests",
        "label": "Pathology_Test",
        "patterns": [
            "core needle biopsy",
            "fine needle aspiration",
            "bone marrow biopsy",
            "surgical pathology",
            "molecular pathology",
            "complete blood count",
            "comprehensive metabolic panel",
            "immunohistochemistry",
            "flow cytometry",
            "immunostaining",
            "molecular testing",
            "cytogenetics",
            "histopathology",
            "NGS panel",
            "biopsy",
            "FNA",
            "cytology",
            "histology",
            "IHC",
            "FISH",
            "CBC",
            "CMP",
        ],
    }
]
with open(pathology_patterns_path, "w", encoding="utf-8") as rule_file:
    json.dump(pathology_patterns, rule_file)

pathology_ruler = EntityRulerInternalApproach()\
    .setInputCols(["splits", "token"])\
    .setOutputCol("pathology_test_rule_chunk")\
    .setPatternsResource(pathology_patterns_path)\
    .setCaseSensitive(False)

# Pathology_Test and NER are alternatives, so merge them here. Grade is
# intentionally excluded: Grade values must coexist with neighboring
# Adverse_Event chunks and are appended as their own split-aligned stream.
chunk_merger = ChunkMergeApproach()\
    .setInputCols([
        "pathology_test_rule_chunk",
        "ner_chunk",
    ])\
    .setOutputCol("base_chunk_all")

ner_stages = [
    documentAssembler,
    document_splitter,
    sentenceDetector,
    tokenizer,
    word_embeddings,
    ner,
    ner_converter_all,
    grade_matcher,
    pathology_ruler,
    chunk_merger,
]

nlpPipeline = Pipeline(stages=ner_stages)
empty_data = spark.createDataFrame([[""]]).toDF("text")
model = nlpPipeline.fit(empty_data)


sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB
[OK!]
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]


In [37]:
from pyspark.sql import functions as F

dataset_path = "/content/dataset/*.txt"

df = (
    spark.sparkContext
    .wholeTextFiles(dataset_path)
    .toDF(["file_path", "text"])
    .withColumn(
        "file_name",
        F.regexp_extract("file_path", r"([^/]+)$", 1)
    )
    .select("file_name", "text")
)

df.orderBy("file_name").show(5, truncate=90)

row_count = df.count()
file_count = df.select("file_name").distinct().count()

print("Rows / complete text files:", row_count)
print("Distinct file names:", file_count)

assert row_count == file_count == 20, (
    f"Expected 20 complete files, got {row_count} rows and "
    f"{file_count} distinct file names."
)

+--------------------+------------------------------------------------------------------------------------------+
|           file_name|                                                                                      text|
+--------------------+------------------------------------------------------------------------------------------+
|oncology_note_01.txt|SYNTHETIC ONCOLOGY FOLLOW-UP NOTE 01\nPatient: Amina R. | Age: 58 | Visit: treatment cy...|
|oncology_note_02.txt|SYNTHETIC ONCOLOGY FOLLOW-UP NOTE 02\nPatient: Daniel M. | Age: 67 | Visit: treatment c...|
|oncology_note_03.txt|SYNTHETIC ONCOLOGY FOLLOW-UP NOTE 03\nPatient: Sofia L. | Age: 45 | Visit: treatment cy...|
|oncology_note_04.txt|SYNTHETIC ONCOLOGY FOLLOW-UP NOTE 04\nPatient: Thomas K. | Age: 72 | Visit: treatment c...|
|oncology_note_05.txt|SYNTHETIC ONCOLOGY FOLLOW-UP NOTE 05\nPatient: Priya S. | Age: 61 | Visit: treatment cy...|
+--------------------+------------------------------------------------------------------

In [38]:
annotated_sdf = model.transform(df)

print(annotated_sdf.columns)

['file_name', 'text', 'document', 'splits', 'sentence', 'token', 'embeddings', 'ner', 'ner_chunk', 'grade_rule_chunk', 'pathology_test_rule_chunk', 'base_chunk_all']


In [41]:
# -------------------------------------------------------------------
# Hybrid candidate assembly, all aligned to the same exploded split row.
#
# `base_chunk_all` is the native ChunkMerge output for Pathology_Test + NER.
# `grade_rule_chunk` remains separate because Grade is an additional entity,
# not an alternative to a neighboring adverse-event mention.
# -------------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SPLIT_COL = "splits"
BASE_CHUNK_COL = "base_chunk_all"
GRADE_CHUNK_COL = "grade_rule_chunk"

def flatten_chunk_column(chunk_column, candidate_source, rule_id=None):
    """Flatten one CHUNK column and retain the parent split's document offset."""
    return (
        annotated_sdf
        .select(
            "file_name",
            F.col(SPLIT_COL).getItem(0).getField("result").alias("split_context"),
            F.col(SPLIT_COL).getItem(0).getField("begin")
                .cast("int")
                .alias("split_begin"),
            F.explode_outer(F.col(chunk_column)).alias("chunk"),
        )
        .where(F.col("chunk").isNotNull())
        .select(
            "file_name",
            "split_context",
            "split_begin",
            F.col("chunk.result").alias("chunks"),
            F.col("chunk.begin").cast("int").alias("begin"),
            F.col("chunk.end").cast("int").alias("end"),
            F.col("chunk.metadata").getItem("sentence").cast("int").alias("sentence_id"),
            F.coalesce(
                F.col("chunk.metadata").getItem("split"),
                F.col("chunk.metadata").getItem("split_id"),
            ).cast("int").alias("split_id"),
            F.col("chunk.metadata").getItem("entity").alias("entities"),
            F.col("chunk.metadata").getItem("confidence").cast("double").alias("confidence"),
            F.lit(candidate_source).alias("candidate_source"),
            F.lit(rule_id).cast("string").alias("rule_id"),
        )
    )

base_candidates = flatten_chunk_column(
    BASE_CHUNK_COL, "merged_chunk"
)
grade_candidates = flatten_chunk_column(
    GRADE_CHUNK_COL, "rule_grade", "grade_regex"
)

# Remove only exact same-label duplicates. Do not collapse a Grade and an
# Adverse_Event simply because they are close together or share a context.
# Do not deduplicate yet: Grade/Pathology rule offsets must first be
# converted from document-relative to split-relative coordinates.
all_candidates = base_candidates.unionByName(grade_candidates)

result_df = all_candidates.toPandas()
if result_df.empty:
    raise RuntimeError("Candidate extraction produced no chunks.")

def normalize_to_split_offsets(row):
    """
    NER / merged chunks use split-relative offsets.
    Rule chunks may use document-relative offsets.
    Prefer the original offset; only convert it when necessary.
    """
    paragraph = "" if pd.isna(row["split_context"]) else str(row["split_context"])
    target = "" if pd.isna(row["chunks"]) else str(row["chunks"])

    try:
        raw_begin = int(row["begin"])
        raw_end = int(row["end"])
        split_begin = int(row["split_begin"])
    except (TypeError, ValueError):
        return pd.Series([pd.NA, pd.NA, "invalid_offsets"])

    def matches_target(begin, end):
        return (
            0 <= begin <= end < len(paragraph)
            and paragraph[begin : end + 1].casefold() == target.casefold()
        )

    # 1. NER / ChunkMerge convention: already relative to split_context.
    if matches_target(raw_begin, raw_end):
        return pd.Series([raw_begin, raw_end, "aligned_local"])

    # 2. Regex/EntityRuler convention: relative to the full document.
    local_begin = raw_begin - split_begin
    local_end = raw_end - split_begin

    if matches_target(local_begin, local_end):
        return pd.Series([local_begin, local_end, "aligned_document_to_local"])

    return pd.Series([raw_begin, raw_end, "unresolved"])


result_df[["begin", "end", "offset_coordinate_status"]] = result_df.apply(
    normalize_to_split_offsets,
    axis=1,
    result_type="expand",
)

valid_offset_statuses = {
    "aligned_local",
    "aligned_document_to_local",
}

invalid_offset_mask = ~result_df["offset_coordinate_status"].isin(
    valid_offset_statuses
)

if invalid_offset_mask.any():
    bad = result_df.loc[
        invalid_offset_mask,
        [
            "file_name", "chunks", "entities", "begin", "end",
            "split_begin", "offset_coordinate_status",
        ],
    ]
    raise RuntimeError(
        f"{len(bad)} candidates have unresolved coordinate systems. "
        f"Examples:\n{bad.head(10).to_string(index=False)}"
    )

# Now that all offsets are in the same coordinate system, safely deduplicate.
source_priority = {
    "rule_grade": 0,
    "rule_pathology": 1,
    "merged_chunk": 2,
}
result_df["_source_priority"] = (
    result_df["candidate_source"].map(source_priority).fillna(3)
)

result_df = (
    result_df
    .sort_values("_source_priority", kind="stable")
    .drop_duplicates(
        ["file_name", "split_context", "begin", "end", "entities"],
        keep="first",
    )
    .drop(columns="_source_priority")
    .reset_index(drop=True)
)

# The chosen pathology-policy terms are rule-locked. Other pathology terms
# remain normal JEV candidates even if they originated from the dictionary.
locked_pathology_terms = {
    "cbc",
    "cmp",
    "complete blood count",
    "comprehensive metabolic panel",
}

pathology_policy_mask = (
    result_df["entities"].eq("Pathology_Test")
    & result_df["chunks"].str.casefold().isin(locked_pathology_terms)
)
result_df.loc[pathology_policy_mask, "candidate_source"] = "rule_pathology"
result_df.loc[pathology_policy_mask, "rule_id"] = "pathology_policy_cbc_cmp"

# If an exact span has a deterministic Grade or CBC/CMP rule, keep the
# policy result rather than a conflicting same-span model label. This does
# not affect adjacent Grade + Adverse_Event spans.
span_keys = ["file_name", "split_context", "begin", "end"]
policy_mask = result_df["candidate_source"].isin(
    ["rule_grade", "rule_pathology"]
)
policy_spans = set(
    result_df.loc[policy_mask, span_keys].itertuples(index=False, name=None)
)
if policy_spans:
    has_policy_at_span = result_df[span_keys].apply(
        lambda row: tuple(row) in policy_spans, axis=1
    )
    result_df = result_df.loc[
        ~has_policy_at_span | policy_mask
    ].copy().reset_index(drop=True)

# This verifies the important invariant before a JEV call: every emitted
# chunk must actually occur in the paragraph supplied as its context.
context_ok = result_df.apply(
    lambda row: str(row["chunks"]).casefold()
    in str(row["split_context"]).casefold(),
    axis=1,
)
if not context_ok.all():
    raise RuntimeError(
        f"{int((~context_ok).sum())} merged chunks are not in their "
        "split_context. Stop and inspect the splitter/rule inputs."
    )

candidate_keys = [
    "file_name", "split_context", "begin", "end", "chunks", "entities"
]
if result_df.duplicated(candidate_keys).any():
    raise RuntimeError(
        "Duplicate exact candidates remain; inspect the split-aligned "
        "candidate assembly before calling JEV."
    )

# Regression gate for the current 20 supplied test notes.
grade_rule_count = int(
    result_df["candidate_source"].eq("rule_grade").sum()
)

print("All hybrid candidates:", len(result_df))
print("Grade-rule candidates:", grade_rule_count)

assert grade_rule_count == 118, (
    f"Grade rule failure: expected 118 candidates, found {grade_rule_count}. "
    "Stop before JEV; inspect grade_rule_chunk / Grade matcher."
)

print("All hybrid candidates:", len(result_df))
print("Grade-rule candidates:", int(result_df["candidate_source"].eq("rule_grade").sum()))
display(result_df["candidate_source"].value_counts(dropna=False))
display(result_df.head(20))


All hybrid candidates: 1209
Grade-rule candidates: 118
All hybrid candidates: 1209
Grade-rule candidates: 118


,count
candidate_source,
merged_chunk,1051
rule_grade,118
rule_pathology,40


,file_name,split_context,split_begin,chunks,begin,end,sentence_id,split_id,entities,confidence,candidate_source,rule_id,offset_coordinate_status
0,oncology_note_01.txt,"Common side effects of FOLFOX with oxaliplatin, leucovorin, and 5-fluorouracil include: dysgeusia (possible adverse event), cold sensitivity (possible adverse event), vomiting (possible adverse ev...",518,Grade 2,1103,1109,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
1,oncology_note_01.txt,"Common side effects of FOLFOX with oxaliplatin, leucovorin, and 5-fluorouracil include: dysgeusia (possible adverse event), cold sensitivity (possible adverse event), vomiting (possible adverse ev...",518,Grade 1,1640,1646,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
2,oncology_note_02.txt,"Common side effects of carboplatin plus paclitaxel include: arthralgias (possible adverse event), constipation (possible adverse event), bone pain (possible adverse event), neutropenia (possible a...",506,Grade 3,483,489,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
3,oncology_note_02.txt,"Common side effects of carboplatin plus paclitaxel include: arthralgias (possible adverse event), constipation (possible adverse event), bone pain (possible adverse event), neutropenia (possible a...",506,Grade 3,634,640,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
4,oncology_note_02.txt,"Common side effects of carboplatin plus paclitaxel include: arthralgias (possible adverse event), constipation (possible adverse event), bone pain (possible adverse event), neutropenia (possible a...",506,Grade 2,908,914,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
5,oncology_note_02.txt,"Common side effects of carboplatin plus paclitaxel include: arthralgias (possible adverse event), constipation (possible adverse event), bone pain (possible adverse event), neutropenia (possible a...",506,grade 1,1058,1064,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
6,oncology_note_02.txt,"Common side effects of carboplatin plus paclitaxel include: arthralgias (possible adverse event), constipation (possible adverse event), bone pain (possible adverse event), neutropenia (possible a...",506,Grade 2,1165,1171,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
7,oncology_note_02.txt,"Common side effects of carboplatin plus paclitaxel include: arthralgias (possible adverse event), constipation (possible adverse event), bone pain (possible adverse event), neutropenia (possible a...",506,Grade 3,1333,1339,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
8,oncology_note_03.txt,"Common side effects of cisplatin with gemcitabine include: dysgeusia (possible adverse event), electrolyte wasting (possible adverse event), nausea (possible adverse event), tinnitus (possible adv...",481,Grade 2,334,340,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local
9,oncology_note_03.txt,"Common side effects of cisplatin with gemcitabine include: dysgeusia (possible adverse event), electrolyte wasting (possible adverse event), nausea (possible adverse event), tinnitus (possible adv...",481,grade 3,484,490,0,NaN,Grade,NaN,rule_grade,grade_regex,aligned_document_to_local


## JEV post-processing: 26-label entity review and scoped assertion

This section starts from `result_df` created above. Spark NLP remains the
high-recall candidate extractor. JEV reviews each extracted span using its
target-marked paragraph context, can confirm/relabel/reject the entity, and
then assigns assertion only to the approved entity types.

The active registry ontology contains the 26 labels agreed for this version.
`Other_Surgery` is normalized to `Cancer_Surgery`; `ICD10_Code` and
`Supportive_Care` are retained in the audit only and excluded from this NER
output. ICD-10 should instead come from a resolver/regex pipeline.

Assertion is intentionally not applied to every entity. The allowed labels
are also constrained by entity type. For example, `Family` is only meaningful
for a diagnosis or clinical condition, and `Planned` is only offered for
treatment/procedure entities. This is safer than forcing semantically
impossible assertion choices.

**Context safety:** `split_context` must be the paragraph containing the
extracted chunk. The code verifies this before any JEV call. Rows that cannot
be aligned to their context are preserved with their NER label and routed to
the review queue instead of being sent to JEV with incorrect context.


In [42]:
# -------------------------------------------------------------------
# Configuration, ontology, and context-safe candidate preparation
# -------------------------------------------------------------------
import json
import re
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typesafe_sdk import Choice, TypeSafeClient

if not globals().get("typesafe_api_key"):
    raise ValueError(
        "Run the Colab Secret setup cell first to create typesafe_api_key."
    )

# Current 26-label cancer-registry ontology.
ACTIVE_ENTITY_LABELS = [
    "Cancer_Dx",
    "Tumor_Finding",
    "Imaging_Test",
    "Pathology_Test",
    "Pathology_Result",
    "Biomarker",
    "Biomarker_Result",
    "Clinical_Condition",
    "Metastasis",
    "Anatomical_Site",
    "Lymph_Node",
    "Histological_Type",
    "Tumor_Size",
    "Staging",
    "Grade",
    "Invasion",
    "Cancer_Surgery",
    "Radiotherapy",
    "Chemotherapy",
    "Targeted_Therapy",
    "Immunotherapy",
    "Hormonal_Therapy",
    "Adverse_Event",
    "Response_To_Treatment",
    "Performance_Status",
    "Date",
]

ENTITY_REVIEW_LABELS = ACTIVE_ENTITY_LABELS + ["NOT_ENTITY"]

# Compatibility with older model outputs. Keep these mappings explicit so
# the source label remains visible in the audit dataframe.
LEGACY_NER_LABEL_MAP = {
    "Other_Surgery": "Cancer_Surgery",
}
OUT_OF_SCOPE_NER_LABELS = {
    "ICD10_Code",
    "Supportive_Care",
}

# Candidate provenance is retained in the audit.  Only narrow, deterministic
# rules are locked before JEV: Grade patterns and the agreed CBC/CMP policy.
# Other dictionary-matched pathology terms still go to JEV for confirmation.
LOCKED_PATHOLOGY_TERMS = {
    "cbc",
    "cmp",
    "complete blood count",
    "comprehensive metabolic panel",
}
GRADE_RULE_PATTERN = re.compile(
    r"(?:grade\s*[1-5]|(?:low|intermediate|high)[-\s]grade)",
    flags=re.IGNORECASE,
)

# Assertion is applied only after JEV produces the final entity label.
ASSERTION_SCOPE = {
    "Cancer_Dx",
    "Tumor_Finding",
    "Clinical_Condition",
    "Metastasis",
    "Staging",
    "Invasion",
    "Cancer_Surgery",
    "Radiotherapy",
    "Chemotherapy",
    "Targeted_Therapy",
    "Immunotherapy",
    "Hormonal_Therapy",
    "Adverse_Event",
    "Response_To_Treatment",
}

ASSERTION_LABELS = [
    "Present",
    "Planned",
    "Past",
    "Family",
    "Absent",
    "Hypothetical",
    "Possible",
]

# Do not offer assertion states that are not meaningful for an entity type.
# This preserves the seven requested labels without forcing bad decisions.
ASSERTION_ALLOWED_BY_ENTITY = {
    "Cancer_Dx": [
        "Present", "Past", "Family", "Absent", "Hypothetical", "Possible"
    ],
    "Tumor_Finding": [
        "Present", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Clinical_Condition": [
        "Present", "Past", "Family", "Absent", "Hypothetical", "Possible"
    ],
    "Metastasis": [
        "Present", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Staging": [
        "Present", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Invasion": [
        "Present", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Cancer_Surgery": [
        "Present", "Planned", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Radiotherapy": [
        "Present", "Planned", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Chemotherapy": [
        "Present", "Planned", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Targeted_Therapy": [
        "Present", "Planned", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Immunotherapy": [
        "Present", "Planned", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Hormonal_Therapy": [
        "Present", "Planned", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Adverse_Event": [
        "Present", "Past", "Absent", "Hypothetical", "Possible"
    ],
    "Response_To_Treatment": [
        "Present", "Past", "Absent", "Hypothetical", "Possible"
    ],
}

# TypeSafe batching and audit thresholds. A JEV choice is never overridden
# by a second hand-written probability threshold; uncertain rows are queued.
MAX_CANDIDATES_PER_CALL = 8
MAX_RETRIES = 2
LOW_JEV_CONFIDENCE = 0.60
LOW_PROBABILITY_MARGIN = 0.15

# Set to True only if an explicit note-section column is guaranteed upstream.
# Paragraph context is required in every case.
REQUIRE_EXPLICIT_SECTION_FOR_ASSERTION = False

ENTITY_CRITERIA = {
    "Cancer_Dx": (
        "A patient cancer diagnosis or malignant disease name. Exclude a "
        "histologic subtype alone, metastatic spread, staging, or a test result."
    ),
    "Tumor_Finding": (
        "A descriptive tumor, lesion, mass, nodule, recurrence, or related "
        "oncologic finding. Exclude the imaging/pathology procedure itself."
    ),
    "Imaging_Test": (
        "An imaging procedure or study, such as CT, MRI, PET, ultrasound, "
        "mammography, or radiograph; not an imaging finding."
    ),
    "Pathology_Test": (
        "A diagnostic, pathology, specimen, cytology, molecular, or monitoring "
        "test; not its result. For this registry, Pathology_Test explicitly "
        "includes CBC, complete blood count, CMP, and comprehensive metabolic "
        "panel, as well as biopsy, cytology, IHC, and molecular pathology tests."
    ),
    "Pathology_Result": (
        "A pathologic interpretation or result, excluding separately labeled "
        "histology, grade, invasion, tumor size, biomarker, or biomarker result."
    ),
    "Biomarker": (
        "A named molecular, receptor, genetic, or laboratory biomarker without "
        "its value or status, for example HER2, PD-L1, or EGFR."
    ),
    "Biomarker_Result": (
        "A biomarker value, expression, mutation status, positivity/negativity, "
        "or other interpreted biomarker result."
    ),
    "Clinical_Condition": (
        "A symptom, sign, clinical condition, or finding not clearly caused by "
        "cancer-directed treatment."
    ),
    "Metastasis": (
        "Metastatic disease, secondary malignant spread, or an explicitly "
        "metastatic site."
    ),
    "Anatomical_Site": (
        "An anatomical body site or location, excluding a lymph-node structure "
        "when Lymph_Node is more specific."
    ),
    "Lymph_Node": (
        "A lymph node, nodal basin, nodal station, or lymph-node involvement."
    ),
    "Histological_Type": (
        "A named histologic or morphologic tumor subtype, such as adenocarcinoma "
        "or squamous-cell carcinoma, when it is not the broader cancer diagnosis."
    ),
    "Tumor_Size": (
        "A tumor dimension, size measurement, or quantitative size description."
    ),
    "Staging": (
        "Cancer stage, TNM component, stage group, or other formal staging value."
    ),
    "Grade": (
        "Tumor grade, differentiation grade, or grade value."
    ),
    "Invasion": (
        "Tumor invasion, margin invasion, lymphovascular/perineural invasion, "
        "or an explicit invasive property."
    ),
    "Cancer_Surgery": (
        "Cancer-directed surgery or resection. Older Other_Surgery predictions "
        "are normalized here in this 26-label ontology."
    ),
    "Radiotherapy": "Radiation therapy or a cancer-directed radiation treatment.",
    "Chemotherapy": "Cytotoxic chemotherapy or a named chemotherapy regimen/drug.",
    "Targeted_Therapy": "Targeted anti-cancer therapy or a named targeted agent.",
    "Immunotherapy": "Cancer immunotherapy or a named immunotherapy agent.",
    "Hormonal_Therapy": "Hormonal or endocrine anti-cancer therapy.",
    "Adverse_Event": (
        "A harmful symptom, complication, or toxicity clearly attributed to "
        "cancer surgery or anti-cancer treatment."
    ),
    "Response_To_Treatment": (
        "An explicit therapeutic response or disease-control assessment, such as "
        "partial response, complete response, stable disease, or progression."
    ),
    "Performance_Status": (
        "Functional performance status, including ECOG, Karnofsky, or an explicit "
        "performance-status assessment."
    ),
    "Date": "A calendar date, date range, or clinically relevant temporal expression.",
    "NOT_ENTITY": (
        "The extracted text is not a valid entity in the active 26-label "
        "cancer-registry ontology."
    ),
}

ASSERTION_CRITERIA = {
    "Present": (
        "Affirmed for the patient as current, observed, active, or ongoing in "
        "the note context. For a treatment, it is being received or currently active."
    ),
    "Planned": (
        "Explicitly intended, scheduled, ordered, or planned for the future; "
        "not yet received/performed."
    ),
    "Past": (
        "Historical, resolved, completed, or previously received/performed; "
        "not current or ongoing."
    ),
    "Family": (
        "Describes a family member or family history, not the patient."
    ),
    "Absent": (
        "Explicitly negated or absent, such as no, without, denies, negative for, "
        "or no evidence of the target entity."
    ),
    "Hypothetical": (
        "Conditional, counterfactual, educational, or hypothetical rather than an "
        "assertion about this patient, for example 'if the patient develops'."
    ),
    "Possible": (
        "Uncertain, suspected, probable, being evaluated, or otherwise not "
        "definitively affirmed or denied."
    ),
}

required_columns = {"file_name", "chunks", "entities", "split_context"}
missing_columns = required_columns - set(result_df.columns)
if missing_columns:
    raise ValueError(
        f"result_df is missing required columns: {sorted(missing_columns)}"
    )

def clean_text(value):
    return "" if pd.isna(value) else str(value).strip()


def optional_int(value):
    if pd.isna(value):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def normalize_ner_label(label):
    label = clean_text(label)
    if label in LEGACY_NER_LABEL_MAP:
        return LEGACY_NER_LABEL_MAP[label]
    if label in OUT_OF_SCOPE_NER_LABELS:
        return None
    return label if label in ACTIVE_ENTITY_LABELS else None


def legacy_label_status(label):
    label = clean_text(label)
    if label in LEGACY_NER_LABEL_MAP:
        return f"mapped:{label}->{LEGACY_NER_LABEL_MAP[label]}"
    if label in OUT_OF_SCOPE_NER_LABELS:
        return f"out_of_scope:{label}"
    if label in ACTIVE_ENTITY_LABELS:
        return "active"
    return f"unsupported:{label or 'blank'}"


def normalize_candidate_source(value):
    value = clean_text(value)
    return value if value else "spark_ner"


def candidate_source_priority(value):
    return {
        "rule_grade": 0,
        "rule_pathology": 1,
        "spark_ner": 2,
    }.get(clean_text(value), 3)


def rule_lock_reason(row):
    """Return a deterministic entity label only for approved narrow rules."""
    source_name = clean_text(row["candidate_source"])
    target = clean_text(row["chunks"])
    normalized_target = target.casefold()
    if (
        source_name == "rule_grade"
        and GRADE_RULE_PATTERN.fullmatch(target) is not None
    ):
        return "grade_regex"
    if (
        source_name == "rule_pathology"
        and normalized_target in LOCKED_PATHOLOGY_TERMS
    ):
        return "pathology_policy_cbc_cmp"
    return None


SECTION_COLUMN_CANDIDATES = [
    "section",
    "section_name",
    "note_section",
    "section_header",
]
SECTION_PATTERN = re.compile(
    r"^\s*(assessment(?:\s+and\s+plan)?|plan|history of present illness|"
    r"hpi|past medical history|family history|interval history|oncology history|"
    r"treatment history|hospital course|impression|diagnosis|pathology|radiology|"
    r"procedure)\s*:?",
    flags=re.IGNORECASE,
)


def infer_section_hint(row):
    for column in SECTION_COLUMN_CANDIDATES:
        if column in row.index:
            value = clean_text(row[column])
            if value:
                return value

    paragraph = clean_text(row["split_context"])
    first_line = paragraph.splitlines()[0] if paragraph else ""
    match = SECTION_PATTERN.match(first_line)
    return match.group(1).strip() if match else "Unknown"


def mark_target_context(paragraph, target, begin=None, end=None):
    """Return target-marked context only when the target is unambiguous."""
    paragraph = clean_text(paragraph)
    target = clean_text(target)

    if not paragraph:
        return "", "missing_context", False
    if not target:
        return paragraph, "missing_target", False

    # Prefer local character offsets when they exactly match the paragraph.
    if (
        begin is not None
        and end is not None
        and 0 <= begin <= end < len(paragraph)
        and paragraph[begin : end + 1] == target
    ):
        marked = (
            paragraph[:begin]
            + "<<TARGET>>"
            + paragraph[begin : end + 1]
            + "<</TARGET>>"
            + paragraph[end + 1 :]
        )
        return marked, "aligned_by_offsets", True

    matches = list(re.finditer(re.escape(target), paragraph, flags=re.IGNORECASE))
    if len(matches) == 1:
        match = matches[0]
        marked = (
            paragraph[: match.start()]
            + "<<TARGET>>"
            + paragraph[match.start() : match.end()]
            + "<</TARGET>>"
            + paragraph[match.end() :]
        )
        return marked, "aligned_by_unique_text_match", True
    if len(matches) > 1:
        return paragraph, "ambiguous_repeated_target", False
    return paragraph, "target_not_found_in_context", False


def probabilities_from_answer(answer):
    return {
        str(label): float(probability)
        for label, probability in answer.probabilities.items()
    }


def probability_margin(probabilities):
    values = sorted(
        (float(value) for value in probabilities.values()), reverse=True
    )
    if len(values) < 2:
        return 1.0
    return values[0] - values[1]


def call_system_one_with_retry(client, state, questions):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return client.system_one(state=state, questions=questions)
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(attempt)
    raise last_error


# Work on a clean, stable index so JEV outputs can be safely joined back.
# Accept an older result_df too, but default its provenance to spark_ner.
result_df_updated = result_df.copy().reset_index(drop=True)
if "candidate_source" not in result_df_updated.columns:
    result_df_updated["candidate_source"] = "spark_ner"
if "rule_id" not in result_df_updated.columns:
    result_df_updated["rule_id"] = pd.NA
result_df_updated["candidate_source"] = result_df_updated[
    "candidate_source"
].map(normalize_candidate_source)

# Deduplicate only identical entity spans. The upstream hybrid assembly has
# already resolved exact rule/model conflicts for Grade and CBC/CMP; do not
# erase distinct clinical labels merely because their offsets coincide.
candidate_dedup_keys = [
    "file_name", "split_context", "begin", "end", "entities"
]
if all(column in result_df_updated.columns for column in candidate_dedup_keys):
    result_df_updated["_candidate_source_priority"] = result_df_updated[
        "candidate_source"
    ].map(candidate_source_priority)
    result_df_updated = (
        result_df_updated
        .sort_values("_candidate_source_priority", kind="stable")
        .drop_duplicates(candidate_dedup_keys, keep="first")
        .drop(columns="_candidate_source_priority")
        .reset_index(drop=True)
    )

result_df_updated["result_row_id"] = np.arange(len(result_df_updated))
result_df_updated["ner_label_original"] = result_df_updated["entities"].map(clean_text)
result_df_updated["ner_confidence"] = pd.to_numeric(
    result_df_updated.get("confidence", np.nan), errors="coerce"
)
result_df_updated["ner_label_for_review"] = result_df_updated[
    "ner_label_original"
].map(normalize_ner_label)
result_df_updated["legacy_label_status"] = result_df_updated[
    "ner_label_original"
].map(legacy_label_status)

context_info = result_df_updated.apply(
    lambda row: mark_target_context(
        row["split_context"],
        row["chunks"],
        optional_int(row["begin"]) if "begin" in row.index else None,
        optional_int(row["end"]) if "end" in row.index else None,
    ),
    axis=1,
    result_type="expand",
)
context_info.columns = [
    "target_context",
    "context_target_status",
    "context_ready",
]
result_df_updated = pd.concat([result_df_updated, context_info], axis=1)
result_df_updated["section_hint"] = result_df_updated.apply(
    infer_section_hint, axis=1
)

result_df_updated["final_entity"] = result_df_updated["ner_label_for_review"]
result_df_updated["entity_retained"] = result_df_updated[
    "final_entity"
].isin(ACTIVE_ENTITY_LABELS)
result_df_updated["jev_entity_choice"] = pd.NA
result_df_updated["jev_entity_confidence"] = np.nan
result_df_updated["jev_entity_margin"] = np.nan
result_df_updated["jev_entity_probabilities"] = None
result_df_updated["jev_entity_status"] = "not_applicable"
result_df_updated["jev_entity_error"] = pd.NA
result_df_updated["entity_needs_review"] = False
result_df_updated["rule_lock_reason"] = result_df_updated.apply(
    rule_lock_reason, axis=1
)
result_df_updated["rule_locked_entity"] = result_df_updated[
    "rule_lock_reason"
].notna()

active_candidate_mask = result_df_updated["ner_label_for_review"].isin(
    ACTIVE_ENTITY_LABELS
)
result_df_updated.loc[active_candidate_mask, "jev_entity_status"] = "pending"
result_df_updated.loc[
    active_candidate_mask & ~result_df_updated["context_ready"],
    "jev_entity_status",
] = "context_unavailable"
result_df_updated.loc[
    active_candidate_mask & ~result_df_updated["context_ready"],
    "entity_needs_review",
] = True
result_df_updated.loc[
    active_candidate_mask & ~result_df_updated["context_ready"],
    "entity_retained",
] = False

# Lock only when the rule's target is safely aligned to its paragraph.
# Otherwise retain the candidate and send it to review rather than trusting
# a rule that was flattened against the wrong split.
locked_entity_mask = (
    active_candidate_mask
    & result_df_updated["rule_locked_entity"]
    & result_df_updated["context_ready"]
)
result_df_updated.loc[locked_entity_mask, "jev_entity_choice"] = (
    result_df_updated.loc[locked_entity_mask, "ner_label_for_review"]
)
result_df_updated.loc[locked_entity_mask, "jev_entity_status"] = "rule_locked"
result_df_updated.loc[locked_entity_mask, "entity_needs_review"] = False

entity_candidate_df = result_df_updated.loc[
    active_candidate_mask
    & result_df_updated["context_ready"]
    & ~locked_entity_mask
].copy()

print("All extracted chunks:", len(result_df_updated))
print("Active ontology candidates:", int(active_candidate_mask.sum()))
print("Deterministic rule-locked candidates:", int(locked_entity_mask.sum()))
print("Context-ready JEV entity candidates:", len(entity_candidate_df))
print(
    "Context-unavailable active candidates:",
    int((active_candidate_mask & ~result_df_updated["context_ready"]).sum()),
)

All extracted chunks: 1209
Active ontology candidates: 1209
Deterministic rule-locked candidates: 158
Context-ready JEV entity candidates: 1051
Context-unavailable active candidates: 0


In [43]:
# -------------------------------------------------------------------
# JEV entity review: confirm, relabel, or reject each NER chunk
# -------------------------------------------------------------------
def score_entity_subset(client, subset):
    state = {"candidates": []}
    questions = {}

    for pos, (_, row) in enumerate(subset.iterrows()):
        state["candidates"].append(
            {
                "target_text": clean_text(row["chunks"]),
                "original_ner_label": clean_text(row["ner_label_for_review"]),
                "section_hint": clean_text(row["section_hint"]),
                "target_context": clean_text(row["target_context"]),
            }
        )

        questions[f"entity_{pos}"] = Choice(
            instructions=(
                f"Classify only `candidates[{pos}].target_text`. The exact "
                "target is delimited by <<TARGET>> and <</TARGET>> inside "
                f"`candidates[{pos}].target_context`. Use the surrounding "
                "paragraph and `section_hint` only to determine the target's "
                "meaning; do not classify a neighboring entity instead.\n\n"
                "Choose exactly one active cancer-registry entity label or "
                "NOT_ENTITY. NOT_ENTITY is appropriate only when the target "
                "itself is not a valid entity in the current ontology."
            ),
            criteria=ENTITY_CRITERIA,
        )

    response = call_system_one_with_retry(client, state, questions)
    scored_rows = []

    for pos, (row_index, row) in enumerate(subset.iterrows()):
        answer = response.answers[f"entity_{pos}"]
        choice = str(answer.choice).strip()
        if choice not in ENTITY_REVIEW_LABELS:
            raise ValueError(f"Unexpected JEV entity label: {choice}")

        probabilities = probabilities_from_answer(answer)
        confidence = float(answer.confidence)
        margin = probability_margin(probabilities)
        original_label = clean_text(row["ner_label_for_review"])

        scored_rows.append(
            {
                "row_index": row_index,
                "choice": choice,
                "confidence": confidence,
                "margin": margin,
                "probabilities": probabilities,
                "final_entity": None if choice == "NOT_ENTITY" else choice,
                "needs_review": (
                    confidence < LOW_JEV_CONFIDENCE
                    or margin < LOW_PROBABILITY_MARGIN
                    or choice != original_label
                ),
            }
        )
    return scored_rows


client = TypeSafeClient(api_key=typesafe_api_key)

entity_groups = entity_candidate_df.groupby(
    ["file_name", "split_context", "section_hint"],
    sort=False,
    dropna=False,
)

for _, paragraph_group in tqdm(
    entity_groups,
    total=entity_groups.ngroups,
    desc="JEV entity review",
):
    for start in range(0, len(paragraph_group), MAX_CANDIDATES_PER_CALL):
        subset = paragraph_group.iloc[start : start + MAX_CANDIDATES_PER_CALL]
        try:
            for item in score_entity_subset(client, subset):
                idx = item["row_index"]
                result_df_updated.at[idx, "jev_entity_choice"] = item["choice"]
                result_df_updated.at[idx, "jev_entity_confidence"] = item[
                    "confidence"
                ]
                result_df_updated.at[idx, "jev_entity_margin"] = item["margin"]
                result_df_updated.at[idx, "jev_entity_probabilities"] = item[
                    "probabilities"
                ]
                result_df_updated.at[idx, "final_entity"] = item["final_entity"]
                result_df_updated.at[idx, "entity_retained"] = (
                    item["final_entity"] in ACTIVE_ENTITY_LABELS
                )
                result_df_updated.at[idx, "entity_needs_review"] = item[
                    "needs_review"
                ]
                result_df_updated.at[idx, "jev_entity_status"] = "scored"
        except Exception as exc:
            # Retain the original normalized NER label if JEV fails.
            for idx in subset.index:
                result_df_updated.at[idx, "jev_entity_status"] = "error"
                result_df_updated.at[idx, "jev_entity_error"] = (
                    f"{type(exc).__name__}: {exc}"
                )
                result_df_updated.at[idx, "entity_needs_review"] = True

entity_input = result_df_updated["ner_label_for_review"].fillna("")
entity_final = result_df_updated["final_entity"].fillna("")
result_df_updated["entity_label_changed"] = entity_input != entity_final

def entity_decision(row):
    status = row["jev_entity_status"]
    if status == "rule_locked":
        return "rule_locked"
    if status == "scored" and not row["entity_retained"]:
        return "rejected"
    if status == "scored" and row["entity_label_changed"]:
        return "relabelled"
    if status == "scored":
        return "confirmed"
    if status == "error":
        return "error_fallback_to_ner"
    if status == "context_unavailable":
        return "context_unavailable_audit_only"
    return row["legacy_label_status"]

result_df_updated["entity_decision"] = result_df_updated.apply(
    entity_decision, axis=1
)

print("\nEntity review summary")
print("Rule locked:", int(result_df_updated["jev_entity_status"].eq("rule_locked").sum()))
print("Scored:", int(result_df_updated["jev_entity_status"].eq("scored").sum()))
print("Relabelled:", int(result_df_updated["entity_decision"].eq("relabelled").sum()))
print("Rejected as NOT_ENTITY:", int(result_df_updated["entity_decision"].eq("rejected").sum()))
print("JEV errors:", int(result_df_updated["jev_entity_status"].eq("error").sum()))


JEV entity review:   0%|          | 0/80 [00:00<?, ?it/s]


Entity review summary
Rule locked: 158
Scored: 1051
Relabelled: 22
Rejected as NOT_ENTITY: 16
JEV errors: 0


In [44]:
# -------------------------------------------------------------------
# JEV assertion: only selected final entity labels receive an assertion
# -------------------------------------------------------------------
result_df_updated["assertion_final"] = "Not_Applicable"
result_df_updated["assertion_allowed_labels"] = None
result_df_updated["jev_assertion_choice"] = pd.NA
result_df_updated["jev_assertion_confidence"] = np.nan
result_df_updated["jev_assertion_margin"] = np.nan
result_df_updated["jev_assertion_probabilities"] = None
result_df_updated["jev_assertion_status"] = "not_applicable"
result_df_updated["jev_assertion_error"] = pd.NA
result_df_updated["assertion_needs_review"] = False

assertion_entity_mask = (
    result_df_updated["entity_retained"]
    & result_df_updated["final_entity"].isin(ASSERTION_SCOPE)
)
assertion_context_mask = result_df_updated["context_ready"].copy()
if REQUIRE_EXPLICIT_SECTION_FOR_ASSERTION:
    assertion_context_mask &= result_df_updated["section_hint"].ne("Unknown")

result_df_updated.loc[assertion_entity_mask, "assertion_final"] = "Unknown"
result_df_updated.loc[
    assertion_entity_mask, "jev_assertion_status"
] = "pending"
result_df_updated.loc[
    assertion_entity_mask, "assertion_allowed_labels"
] = result_df_updated.loc[assertion_entity_mask, "final_entity"].map(
    ASSERTION_ALLOWED_BY_ENTITY
)

unavailable_assertion_context = assertion_entity_mask & ~assertion_context_mask
result_df_updated.loc[
    unavailable_assertion_context, "jev_assertion_status"
] = "context_unavailable"
result_df_updated.loc[
    unavailable_assertion_context, "assertion_needs_review"
] = True

assertion_candidate_df = result_df_updated.loc[
    assertion_entity_mask & assertion_context_mask
].copy()

def score_assertion_subset(client, subset):
    state = {"candidates": []}
    questions = {}

    for pos, (_, row) in enumerate(subset.iterrows()):
        allowed_labels = list(row["assertion_allowed_labels"])
        state["candidates"].append(
            {
                "target_text": clean_text(row["chunks"]),
                "final_entity_label": clean_text(row["final_entity"]),
                "section_hint": clean_text(row["section_hint"]),
                "target_context": clean_text(row["target_context"]),
                "allowed_assertion_labels": allowed_labels,
            }
        )
        criteria = {
            label: ASSERTION_CRITERIA[label] for label in allowed_labels
        }
        questions[f"assertion_{pos}"] = Choice(
            instructions=(
                f"Determine the assertion status only for "
                f"`candidates[{pos}].target_text`, whose final entity type is "
                f"`candidates[{pos}].final_entity_label`. The target is marked "
                "with <<TARGET>> and <</TARGET>> in "
                f"`candidates[{pos}].target_context`. Use the paragraph and "
                "section hint to determine whether the statement refers to the "
                "patient, its timing, certainty, negation, or plan.\n\n"
                "Choose exactly one of the allowed assertion labels. Do not infer "
                "an assertion from a neighboring entity."
            ),
            criteria=criteria,
        )

    response = call_system_one_with_retry(client, state, questions)
    scored_rows = []

    for pos, (row_index, row) in enumerate(subset.iterrows()):
        answer = response.answers[f"assertion_{pos}"]
        choice = str(answer.choice).strip()
        allowed_labels = set(row["assertion_allowed_labels"])
        if choice not in allowed_labels:
            raise ValueError(
                f"Unexpected JEV assertion label for row {row_index}: {choice}"
            )

        probabilities = probabilities_from_answer(answer)
        confidence = float(answer.confidence)
        margin = probability_margin(probabilities)
        scored_rows.append(
            {
                "row_index": row_index,
                "choice": choice,
                "confidence": confidence,
                "margin": margin,
                "probabilities": probabilities,
                "needs_review": (
                    confidence < LOW_JEV_CONFIDENCE
                    or margin < LOW_PROBABILITY_MARGIN
                ),
            }
        )
    return scored_rows


assertion_groups = assertion_candidate_df.groupby(
    ["file_name", "split_context", "section_hint"],
    sort=False,
    dropna=False,
)

for _, paragraph_group in tqdm(
    assertion_groups,
    total=assertion_groups.ngroups,
    desc="JEV assertion",
):
    for start in range(0, len(paragraph_group), MAX_CANDIDATES_PER_CALL):
        subset = paragraph_group.iloc[start : start + MAX_CANDIDATES_PER_CALL]
        try:
            for item in score_assertion_subset(client, subset):
                idx = item["row_index"]
                result_df_updated.at[idx, "jev_assertion_choice"] = item[
                    "choice"
                ]
                result_df_updated.at[idx, "jev_assertion_confidence"] = item[
                    "confidence"
                ]
                result_df_updated.at[idx, "jev_assertion_margin"] = item[
                    "margin"
                ]
                result_df_updated.at[idx, "jev_assertion_probabilities"] = item[
                    "probabilities"
                ]
                result_df_updated.at[idx, "assertion_final"] = item["choice"]
                result_df_updated.at[idx, "assertion_needs_review"] = item[
                    "needs_review"
                ]
                result_df_updated.at[idx, "jev_assertion_status"] = "scored"
        except Exception as exc:
            for idx in subset.index:
                result_df_updated.at[idx, "jev_assertion_status"] = "error"
                result_df_updated.at[idx, "jev_assertion_error"] = (
                    f"{type(exc).__name__}: {exc}"
                )
                result_df_updated.at[idx, "assertion_needs_review"] = True

result_df_updated["needs_review"] = (
    result_df_updated["entity_needs_review"]
    | result_df_updated["assertion_needs_review"]
)

print("\nAssertion summary")
print("Applicable final entities:", int(assertion_entity_mask.sum()))
print("Scored:", int(result_df_updated["jev_assertion_status"].eq("scored").sum()))
print("Context unavailable:", int(unavailable_assertion_context.sum()))
print("JEV errors:", int(result_df_updated["jev_assertion_status"].eq("error").sum()))

JEV assertion:   0%|          | 0/79 [00:00<?, ?it/s]


Assertion summary
Applicable final entities: 1009
Scored: 1009
Context unavailable: 0
JEV errors: 0


In [45]:
# -------------------------------------------------------------------
# Auditable final registry dataframe, review queue, and exports
# -------------------------------------------------------------------
# Keep rejected, out-of-scope, and context-unsafe candidates in the audit,
# but never include them in the final registry entity output. A candidate
# without a verified paragraph is evidence for review, not a valid entity.
audit_df = result_df_updated.copy()
registry_result_df = audit_df.loc[audit_df["entity_retained"]].copy()
registry_result_df["entities"] = registry_result_df["final_entity"]
registry_result_df["assertion"] = registry_result_df["assertion_final"]

final_dedup_keys = [
    "file_name", "split_context", "begin", "end", "entities"
]
duplicate_final_count = int(
    registry_result_df.duplicated(final_dedup_keys, keep="first").sum()
)
if duplicate_final_count:
    registry_result_df = registry_result_df.drop_duplicates(
        final_dedup_keys, keep="first"
    ).copy()
    print(
        "Safety guard removed", duplicate_final_count,
        "unexpected duplicate final rows; inspect audit provenance."
    )

audit_columns = [
    "file_name",
    "chunks",
    "begin",
    "end",
    "sentence_id",
    "split_id",
    "section_hint",
    "split_context",
    "candidate_source",
    "rule_id",
    "rule_lock_reason",
    "rule_locked_entity",
    "ner_label_original",
    "ner_label_for_review",
    "ner_confidence",
    "final_entity",
    "entity_decision",
    "jev_entity_choice",
    "jev_entity_confidence",
    "jev_entity_margin",
    "assertion_final",
    "jev_assertion_confidence",
    "jev_assertion_margin",
    "needs_review",
    "context_target_status",
    "jev_entity_status",
    "jev_assertion_status",
    "jev_entity_error",
    "jev_assertion_error",
]
audit_columns = [column for column in audit_columns if column in audit_df.columns]

review_queue_df = audit_df.loc[audit_df["needs_review"], audit_columns].copy()

print("\nFinal registry rows:", len(registry_result_df))
print("Audit rows:", len(audit_df))
print("Review queue rows:", len(review_queue_df))

print("\nEntity labels changed by JEV")
display(
    audit_df.loc[
        audit_df["entity_decision"].isin(["relabelled", "rejected"]),
        audit_columns,
    ]
    .sort_values(
        ["needs_review", "jev_entity_confidence"],
        ascending=[False, True],
    )
    .head(100)
)

print("\nAssertion review queue")
display(
    review_queue_df.sort_values(
        ["jev_assertion_confidence", "jev_entity_confidence"],
        ascending=[True, True],
    ).head(100)
)

def json_safe_export(dataframe):
    exported = dataframe.copy()
    json_columns = [
        "jev_entity_probabilities",
        "jev_assertion_probabilities",
        "assertion_allowed_labels",
    ]
    for column in json_columns:
        if column in exported.columns:
            exported[column] = exported[column].apply(
                lambda value: (
                    json.dumps(value, ensure_ascii=False)
                    if isinstance(value, (dict, list))
                    else None
                )
            )
    return exported.where(pd.notna(exported), None)


output_prefix = "/content/cancer_registry_jev"
audit_export_df = json_safe_export(audit_df)
registry_export_df = json_safe_export(registry_result_df)
review_export_df = json_safe_export(review_queue_df)

audit_export_df.to_csv(f"{output_prefix}_audit.csv", index=False)
audit_export_df.to_json(
    f"{output_prefix}_audit.json",
    orient="records",
    indent=2,
    force_ascii=False,
)
registry_export_df.to_csv(f"{output_prefix}_final.csv", index=False)
registry_export_df.to_json(
    f"{output_prefix}_final.json",
    orient="records",
    indent=2,
    force_ascii=False,
)
review_export_df.to_csv(f"{output_prefix}_review_queue.csv", index=False)

print("\nSaved:")
print(f"{output_prefix}_audit.csv")
print(f"{output_prefix}_audit.json")
print(f"{output_prefix}_final.csv")
print(f"{output_prefix}_final.json")
print(f"{output_prefix}_review_queue.csv")

display(
    registry_result_df[
        [
            "file_name",
            "chunks",
            "entities",
            "assertion",
            "ner_label_original",
            "entity_decision",
            "needs_review",
        ]
    ].head(30)
)



Final registry rows: 1193
Audit rows: 1209
Review queue rows: 241

Entity labels changed by JEV


,file_name,chunks,begin,end,sentence_id,split_id,section_hint,split_context,candidate_source,rule_id,...,jev_entity_margin,assertion_final,jev_assertion_confidence,jev_assertion_margin,needs_review,context_target_status,jev_entity_status,jev_assertion_status,jev_entity_error,jev_assertion_error
1049,oncology_note_18.txt,complication,300,311,3,NaN,Interval history,"Interval history:\nThe patient is receiving trastuzumab plus pertuzumab. Since the last visit, symptoms began or worsened after the most recent targeted therapy treatment. The patient denies new c...",merged_chunk,None,...,0.00,Not_Applicable,NaN,NaN,True,aligned_by_offsets,scored,not_applicable,<NA>,<NA>
1193,oncology_note_20.txt,ipilimumab,1554,1563,9,NaN,Unknown,"Common side effects of nivolumab plus ipilimumab include: abdominal pain (possible adverse event), immune-mediated diarrhea (possible adverse event), shortness of breath (possible adverse event), ...",merged_chunk,None,...,0.01,Present,0.97,0.97,True,aligned_by_offsets,scored,scored,<NA>,<NA>
1205,oncology_note_20.txt,ipilimumab,2006,2015,12,NaN,Unknown,"Common side effects of nivolumab plus ipilimumab include: abdominal pain (possible adverse event), immune-mediated diarrhea (possible adverse event), shortness of breath (possible adverse event), ...",merged_chunk,None,...,0.04,Present,0.66,0.46,True,aligned_by_offsets,scored,scored,<NA>,<NA>
296,oncology_note_03.txt,chemotherapy-related adverse event,554,587,2,NaN,Unknown,"Common side effects of cisplatin with gemcitabine include: dysgeusia (possible adverse event), electrolyte wasting (possible adverse event), nausea (possible adverse event), tinnitus (possible adv...",merged_chunk,None,...,0.00,Not_Applicable,NaN,NaN,True,aligned_by_offsets,scored,not_applicable,<NA>,<NA>
544,oncology_note_08.txt,ipilimumab,808,817,3,NaN,Unknown,"Common side effects of nivolumab plus ipilimumab include: immune-mediated diarrhea (possible adverse event), shortness of breath (possible adverse event), fatigue (possible adverse event), hypophy...",merged_chunk,None,...,0.03,Present,0.85,0.77,True,aligned_by_offsets,scored,scored,<NA>,<NA>
378,oncology_note_05.txt,fever,234,238,3,NaN,Interval history,"Interval history:\nThe patient is receiving sorafenib. Since the last visit, symptoms began or worsened after the most recent targeted therapy treatment. The patient denies new cancer-related foca...",merged_chunk,None,...,0.04,Absent,1.00,1.00,True,aligned_by_offsets,scored,scored,<NA>,<NA>
540,oncology_note_08.txt,ipilimumab,664,673,2,NaN,Unknown,"Common side effects of nivolumab plus ipilimumab include: immune-mediated diarrhea (possible adverse event), shortness of breath (possible adverse event), fatigue (possible adverse event), hypophy...",merged_chunk,None,...,0.05,Present,0.96,0.95,True,aligned_by_offsets,scored,scored,<NA>,<NA>
562,oncology_note_08.txt,ipilimumab,1635,1644,9,NaN,Unknown,"Common side effects of nivolumab plus ipilimumab include: immune-mediated diarrhea (possible adverse event), shortness of breath (possible adverse event), fatigue (possible adverse event), hypophy...",merged_chunk,None,...,0.06,Present,0.81,0.72,True,aligned_by_offsets,scored,scored,<NA>,<NA>
1176,oncology_note_20.txt,nivolumab,774,782,3,NaN,Unknown,"Common side effects of nivolumab plus ipilimumab include: abdominal pain (possible adverse event), immune-mediated diarrhea (possible adverse event), shortness of breath (possible adverse event), ...",merged_chunk,None,...,0.04,Present,0.88,0.81,True,aligned_by_offsets,scored,scored,<NA>,<NA>
510,oncology_note_07.txt,adverse event,1307,1319,8,NaN,Unknown,"Common side effects of pembrolizumab include: arthralgia (possible adverse event), fatigue (possible adverse event), cough (possible adverse event), pruritus (possible adverse event), colitis (pos...",merged_chunk,None,...,0.05,Not_Applicable,NaN,NaN,True,aligned_by_offsets,scored,not_applicable,<NA>,<NA>



Assertion review queue


,file_name,chunks,begin,end,sentence_id,split_id,section_hint,split_context,candidate_source,rule_id,...,jev_entity_margin,assertion_final,jev_assertion_confidence,jev_assertion_margin,needs_review,context_target_status,jev_entity_status,jev_assertion_status,jev_entity_error,jev_assertion_error
590,oncology_note_09.txt,external-beam pelvic radiotherapy,618,650,2,NaN,Unknown,"Common side effects of external-beam pelvic radiotherapy include: fatigue (possible adverse event), vaginal irritation (possible adverse event), moist desquamation (possible adverse event), skin e...",merged_chunk,None,...,1.00,Possible,0.32,0.06,True,aligned_by_offsets,scored,scored,<NA>,<NA>
414,oncology_note_05.txt,mucositis,1404,1412,9,NaN,Unknown,"Common side effects of sorafenib include: fatigue (possible adverse event), diarrhea (possible adverse event), hypertension (possible adverse event), rash (possible adverse event), hoarseness (pos...",merged_chunk,None,...,0.93,Present,0.33,0.04,True,aligned_by_offsets,scored,scored,<NA>,<NA>
212,oncology_note_01.txt,nausea,1648,1653,9,NaN,Unknown,"Common side effects of FOLFOX with oxaliplatin, leucovorin, and 5-fluorouracil include: dysgeusia (possible adverse event), cold sensitivity (possible adverse event), vomiting (possible adverse ev...",merged_chunk,None,...,0.96,Possible,0.33,0.08,True,aligned_by_offsets,scored,scored,<NA>,<NA>
408,oncology_note_05.txt,pruritus,1117,1124,7,NaN,Unknown,"Common side effects of sorafenib include: fatigue (possible adverse event), diarrhea (possible adverse event), hypertension (possible adverse event), rash (possible adverse event), hoarseness (pos...",merged_chunk,None,...,0.98,Present,0.33,0.00,True,aligned_by_offsets,scored,scored,<NA>,<NA>
440,oncology_note_06.txt,nausea,238,243,0,NaN,Unknown,"Common side effects of trastuzumab plus pertuzumab include: fatigue (possible adverse event), infusion chills (possible adverse event), fever (possible adverse event), headache (possible adverse e...",merged_chunk,None,...,0.91,Past,0.34,0.03,True,aligned_by_offsets,scored,scored,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,oncology_note_06.txt,chills,103,108,0,NaN,Unknown,"Common side effects of trastuzumab plus pertuzumab include: fatigue (possible adverse event), infusion chills (possible adverse event), fever (possible adverse event), headache (possible adverse e...",merged_chunk,None,...,0.90,Past,0.50,0.23,True,aligned_by_offsets,scored,scored,<NA>,<NA>
1055,oncology_note_18.txt,headache,137,144,0,NaN,Unknown,"Common side effects of trastuzumab plus pertuzumab include: peripheral edema (possible adverse event), fatigue (possible adverse event), headache (possible adverse event), decreased appetite (poss...",merged_chunk,None,...,0.89,Present,0.50,0.24,True,aligned_by_offsets,scored,scored,<NA>,<NA>
175,oncology_note_01.txt,reduced appetite,247,262,0,NaN,Unknown,"Common side effects of FOLFOX with oxaliplatin, leucovorin, and 5-fluorouracil include: dysgeusia (possible adverse event), cold sensitivity (possible adverse event), vomiting (possible adverse ev...",merged_chunk,None,...,0.93,Present,0.50,0.21,True,aligned_by_offsets,scored,scored,<NA>,<NA>
204,oncology_note_01.txt,FOLFOX,1374,1379,7,NaN,Unknown,"Common side effects of FOLFOX with oxaliplatin, leucovorin, and 5-fluorouracil include: dysgeusia (possible adverse event), cold sensitivity (possible adverse event), vomiting (possible adverse ev...",merged_chunk,None,...,0.96,Past,0.50,0.20,True,aligned_by_offsets,scored,scored,<NA>,<NA>



Saved:
/content/cancer_registry_jev_audit.csv
/content/cancer_registry_jev_audit.json
/content/cancer_registry_jev_final.csv
/content/cancer_registry_jev_final.json
/content/cancer_registry_jev_review_queue.csv


,file_name,chunks,entities,assertion,ner_label_original,entity_decision,needs_review
0,oncology_note_01.txt,Grade 2,Grade,Not_Applicable,Grade,rule_locked,False
1,oncology_note_01.txt,Grade 1,Grade,Not_Applicable,Grade,rule_locked,False
2,oncology_note_02.txt,Grade 3,Grade,Not_Applicable,Grade,rule_locked,False
3,oncology_note_02.txt,Grade 3,Grade,Not_Applicable,Grade,rule_locked,False
4,oncology_note_02.txt,Grade 2,Grade,Not_Applicable,Grade,rule_locked,False
5,oncology_note_02.txt,grade 1,Grade,Not_Applicable,Grade,rule_locked,False
6,oncology_note_02.txt,Grade 2,Grade,Not_Applicable,Grade,rule_locked,False
7,oncology_note_02.txt,Grade 3,Grade,Not_Applicable,Grade,rule_locked,False
8,oncology_note_03.txt,Grade 2,Grade,Not_Applicable,Grade,rule_locked,False
9,oncology_note_03.txt,grade 3,Grade,Not_Applicable,Grade,rule_locked,False


# Cancer registry pipeline — all applied fixes

Use `CancerRegistry_hybrid_all_assertion_all_fixes.ipynb` as the replacement
notebook. It starts from the original notebook and contains the changes below.

## 1. Fix the repeated CBC/CMP rows before JEV

Both rule stages now consume the paragraph-level `splits` annotation:

```python
.setInputCols(["splits", "token"])
```

They do **not** consume the original full-document annotation. The three
split-aligned CHUNK outputs are then merged inside the Spark pipeline:

```python
ChunkMergeApproach() \
    .setInputCols(["grade_rule_chunk", "pathology_test_rule_chunk", "ner_chunk"]) \
    .setOutputCol("ner_chunk_all")
```

Only `ner_chunk_all` is flattened into `result_df`. This prevents a
document-level rule chunk from being paired with every paragraph in the note,
and avoids manual Pandas/Spark union-and-deduplication logic.

The notebook stops before JEV if any rule candidate is not found in its own
`split_context`, or if exact duplicate rule candidates remain.

## 2. Grade rule

The notebook adds the narrow rule:

```text
grade 1–5, low-grade, intermediate-grade, high-grade
```

Matched Grade candidates are retained as deterministic entities and recorded
with `candidate_source=rule_grade` and `entity_decision=rule_locked`.

## 3. Pathology/Test rule

The split-aligned `EntityRulerInternalApproach` dictionary includes the updated
pathology/test terms, including biopsy/cytology/IHC/molecular terms and:

```text
CBC, complete blood count, CMP, comprehensive metabolic panel
```

The merge stage resolves overlapping chunks; longer, more informative spans are
preferred. For example, `core needle biopsy` is retained instead of a nested
`biopsy` span.

## 4. CBC/CMP policy in JEV

For this registry version, `CBC`, `CMP`, `complete blood count`, and
`comprehensive metabolic panel` are locked as `Pathology_Test`. They do not go
to JEV for a potentially conflicting label decision. Other pathology dictionary
matches still go to JEV for confirmation or relabelling.

The JEV `Pathology_Test` criterion was also updated so it explicitly states
this policy. That makes the remaining JEV decisions consistent with the
ontology.

## 5. Grade assertion removed

`Grade` is no longer in `ASSERTION_SCOPE` or `ASSERTION_ALLOWED_BY_ENTITY`.
Its final assertion is always `Not_Applicable`; no JEV assertion call is made
for Grade. All other agreed assertion entities remain unchanged.

## 6. Context safety

Every candidate must be aligned to its containing paragraph by split-local
offsets or one unique text occurrence before JEV is called. A candidate without
verified context is preserved in the audit/review queue only; it is not emitted
as a final registry row.

## 7. Candidate provenance and final safeguards

The audit export now includes:

- `candidate_source` (`merged_chunk`, `rule_grade`, or `rule_pathology`)
- `rule_id`
- `rule_lock_reason`
- `rule_locked_entity`

Exact same-span conflict resolution occurs in `ChunkMergeApproach`: the
leftmost input wins for an exact tie, so Grade and pathology rules are listed
before the general NER chunks. A final exact-row deduplication guard is retained
as a last safety check, but it should report zero removals in a correct run.

## 8. Reference/evaluation update

The LLM-referee builders now remove assertion references for Grade while
retaining Grade entity references. Re-evaluating run 3 under that correct scope
gave assertion conditional accuracy **85.70%** and relaxed end-to-end
entity-plus-assertion F1 **76.60%**. The 160 unmapped rows in the old run are
the known CBC/CMP paragraph-cross-product rows and should not be treated as a
valid post-fix metric.

## Run order

1. Upload/open the fixed notebook in Colab.
2. Run the dependency, license, key, model, and input-text cells as before.
3. Run the new candidate-extraction cells.
4. Confirm the `candidate_source` counts and that no alignment exception was
   raised.
5. Run JEV entity review and then JEV assertion.
6. Use the audit CSV for provenance and the final CSV for registry output.
